# Part 1: Làm sạch & Tích hợp Dữ liệu Đa nguồn
**Thành viên phụ trách:** Thành viên 1 | **Nhánh Git:** `feature/cleaning-integration`

Notebook trình bày: (1) đánh giá dữ liệu thiếu / sai định dạng, (2) khử trùng lặp 3 cấp bằng **hàm băm SHA-256**,
(3) tích hợp chuyến bay đi/đến + thời tiết + tải sân bay, (4) kiểm tra tính nhất quán sau tích hợp (Cohen's Kappa),
(5) đối chiếu tỷ lệ trễ/OTP các hãng với số liệu Cục Hàng không (CAAV).

## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "config.py").exists():
    ROOT = ROOT.parent          # notebook nằm trong notebooks/
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

import config
import pipeline.processor as P
from pipeline.transformation import apply_labels

THRESHOLD = 15
HAS_RAW = any((ROOT / "data" / "raw").glob("*.csv")) or (ROOT / "data" / "debug_data").exists()
FIG_DIR = ROOT.parent / "tv1_figures"          # ngoài git
FIG_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.dpi"] = 110

def save(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

print("Gốc repo:", ROOT)
print("Có dữ liệu thô (data/raw hoặc debug_data):", HAS_RAW)

## 2. Đọc & tích hợp dữ liệu đa nguồn
`P.prepare_dataset()` đọc `data/raw/*.csv` (5 bảng FIDS sân bay + Aviationstack API), chuẩn hóa về schema chung `FIELDS`,
khử trùng lặp 3 cấp và ghép thời tiết Open-Meteo + tải sân bay theo `(airport, giờ bay)`.

In [ ]:
if HAS_RAW:
    side, leg, stats, per_src = P.prepare_dataset()
    print(f"Snapshot thô:                 {stats['snapshots']:,}")
    print(f"Trùng tuyệt đối (raw_hash):   {stats['exact_duplicates']:,}")
    print(f"Sau cấp 0 (raw_hash):         {stats['after_exact_dedup']:,}")
    print(f"Sau cấp 1 (event_hash):       {stats['after_side_dedup']:,}")
    print(f"Sau cấp 2 (leg_hash):         {stats['after_leg_dedup']:,}")
    print(f"Chuyến thấy ở cả 2 phía:      {stats['legs_seen_both_sides']:,}")
else:
    print("Không có data/raw -> các mục 3-5 bị bỏ qua, mục 6-7 đọc thẳng FILE_CLEANED.")

## 3. Dữ liệu thiếu & sai định dạng (dữ liệu THÔ, trước làm sạch)
Quy tắc định dạng: `flight_date` = `YYYY-MM-DD`; `scheduled_time` = `HH:MM`; `flight_no` = mã hãng 2 ký tự + 1–4 chữ số;
`direction` ∈ {DEPARTURES, ARRIVALS}; `airport` = mã IATA 3 chữ cái.

In [ ]:
if HAS_RAW:
    fr = stats["format_report"]
    display(fr)
    ax = fr.set_index("Cột")["Thiếu (%)"].sort_values().plot(kind="barh", figsize=(7, 4), color="#3b82f6")
    ax.set_xlabel("Tỷ lệ thiếu (%)"); ax.set_title("Tỷ lệ thiếu theo cột - dữ liệu thô")
    save("01_missing_raw")

In [ ]:
if HAS_RAW:
    print("Tên sân bay đầu kia chưa ánh xạ được mã IATA (top 20):")
    display(stats["unmapped_top"].to_frame("Số bản ghi"))

**Cách xử lý:**
- `actual_time` trống nhưng trạng thái đã cất/hạ cánh (`Departed`, `Arrived`, `Landed`…) → điền từ `estimated_time` (giờ cập nhật trên FIDS).
- `estimated_time` / `actual_time` trống ở chuyến chưa bay là **thiếu hợp lệ** (chưa xảy ra), không điền.
- Chuẩn hóa in hoa + bỏ khoảng trắng các cột khóa; số hiệu chuyến bỏ khoảng trắng và số 0 đứng đầu (`VN 0213` → `VN213`).
- Tên sân bay đầu kia (`Ha Noi (HAN)`, `Da Nang`…) được ánh xạ sang mã IATA; bản ghi không ánh xạ được vẫn giữ nhưng dùng khóa dự phòng.
- Chỉ giữ 7 hãng Việt Nam (`VN, VJ, QH, BL, VU, 9G, 0V`); tải sân bay được tính **trước** khi lọc (tính trên mọi hãng).

## 4. Khử trùng lặp bằng hàm băm SHA-256
Mỗi cấp nối các cột khóa đã chuẩn hóa thành 1 chuỗi `A|B|C…`, băm SHA-256 và lấy 16 ký tự hex đầu (64 bit).
Hai bản ghi trùng khóa ⇔ trùng mã băm, nên chỉ cần so 1 cột thay vì nhiều cột.

| Cấp | Cột băm | Khóa băm | Quy tắc giữ bản ghi |
|---|---|---|---|
| 0 – trùng tuyệt đối | `raw_hash` | cả 12 cột `FIELDS` | giữ 1 bản (bản sau cùng) |
| 1 – cùng phía sân bay | `event_hash` | `airport, direction, flight_date, flight_no, scheduled_time` | giàu thông tin nhất (`has_info`, `has_done_act`) rồi mới nhất (`scraped_at`) |
| 2 – cấp chuyến bay | `leg_hash` | `leg_key` = `flight_no|flight_date|origin>destination` | gộp bảng đi + bảng đến; ưu tiên giàu thông tin, sân bay có theo dõi, phía DEPARTURES, mới nhất |

In [ ]:
if HAS_RAW:
    display(side[["flight_no", "flight_date", "airport", "direction", "scheduled_time", "source",
                  "raw_hash", "event_hash", "leg_hash", "leg_key"]].head(5))
    funnel = pd.Series({
        "Snapshot thô": stats["snapshots"],
        "Cấp 0: raw_hash": stats["after_exact_dedup"],
        "Cấp 1: event_hash\n(hãng VN)": stats["after_side_dedup"],
        "Cấp 2: leg_hash": stats["after_leg_dedup"],
    }).astype(int)
    display(funnel.to_frame("Số dòng"))
    ax = funnel.plot(kind="bar", figsize=(8, 4), color=["#94a3b8", "#64748b", "#3b82f6", "#1d4ed8"])
    for i, v in enumerate(funnel):
        ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
    ax.set_ylabel("Số dòng"); ax.set_title("Số dòng qua từng cấp khử trùng"); plt.xticks(rotation=15)
    save("02_dedup_funnel")

## 5. Kiểm tra tính nhất quán sau tích hợp

In [ ]:
if HAS_RAW:
    ir = P.integration_report(leg)
    display(ir)

In [ ]:
if HAS_RAW:
    side_l = apply_labels(side, THRESHOLD)
    sv = side_l[side_l["status_clean"].isin(P.LABELS)].assign(is_cx=lambda d: d["status_clean"].eq("cancelled"))
    g = sv.groupby("leg_key")["is_cx"].agg(["nunique", "size"])
    multi = g[g["size"] > 1]
    print(f"Nhất quán hủy/không hủy giữa các bản ghi cùng leg_key: "
          f"{(multi['nunique'] == 1).mean() * 100:.1f}% trên {len(multi):,} chuyến có >= 2 bản ghi")

**Cohen's Kappa** đo mức đồng thuận nhãn (`on_time / delayed / cancelled`) giữa 2 nguồn trên cùng sự kiện,
đã loại phần trùng do ngẫu nhiên: κ = (p_o − p_e) / (1 − p_e), với p_o = tỷ lệ trùng nhãn quan sát, p_e = tỷ lệ trùng kỳ vọng nếu 2 nguồn độc lập.
Cách đọc (Landis & Koch): < 0.2 kém, 0.21–0.40 khá thấp, 0.41–0.60 trung bình, 0.61–0.80 tốt, > 0.80 rất tốt.

In [ ]:
if HAS_RAW:
    sa = P.source_agreement(per_src, THRESHOLD)
    display(sa if not sa.empty else "Chưa có cặp nguồn nào đủ >= 20 sự kiện chung.")

## 6. Dữ liệu sạch & đa nguồn

In [ ]:
if HAS_RAW:
    df, rules = P.label_and_report(side, leg, stats, THRESHOLD, per_src)
    df.to_csv(config.FILE_CLEANED, index=False, encoding="utf-8-sig")
    print(f"Đã ghi {config.FILE_CLEANED.name}: {df.shape[0]:,} dòng x {df.shape[1]} cột")
else:
    df = pd.read_csv(config.FILE_CLEANED)
    print(f"Đọc {config.FILE_CLEANED.name}: {df.shape[0]:,} dòng x {df.shape[1]} cột")
assert list(df.columns) == config.CLEANED_FIELDS and df["leg_key"].is_unique
display(df["status_clean"].value_counts().to_frame("Số chuyến"))

In [ ]:
src = df["source"].value_counts()
display(src.to_frame("Số chuyến"))
ax = src.plot(kind="bar", figsize=(8, 4), color="#3b82f6")
for i, v in enumerate(src):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Số chuyến"); ax.set_title(f"Số chuyến theo nguồn dữ liệu ({src.size} nguồn)"); plt.xticks(rotation=20)
save("03_source_counts")

## 7. Tỷ lệ trễ / OTP các hãng so với CAAV

In [ ]:
al = P.airline_table(df)
display(al)
pl = al[al["Hãng"] != "ALL"].sort_values("Trễ (%)", ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(pl))
ax.bar(x - 0.2, pl["Trễ (%)"], 0.4, label="Trễ (%)", color="#f59e0b")
ax.bar(x + 0.2, pl["OTP (%)"], 0.4, label="OTP (%)", color="#10b981")
ax.set_xticks(x, pl["Hãng"]); ax.set_ylabel("%"); ax.legend()
ax.set_title(f"Tỷ lệ trễ và OTP theo hãng (ngưỡng {THRESHOLD} phút)")
save("04_airline_delay_otp")

**Kết luận & bàn giao cho TV2:**
- `data/flights_cleaned_global.csv` đúng 24 cột theo `CLEANED_FIELDS`, 1 dòng = 1 chặng bay (`leg_key` duy nhất), đã ghép thời tiết và tải sân bay.
- Các cột băm (`raw_hash`, `event_hash`, `leg_hash`) chỉ là cột trung gian, không xuất ra file.
- TV2 dùng file này cho `check_data.py` (data contract) và bước biến đổi dữ liệu.